# Notebook 71 — Offline Replay Specifies Deployment Readiness

This notebook continues the adaptive infrastructure series:

- Notebook 61: telemetry rows
- Notebook 67: policy geometry
- Notebook 71: offline replay evaluation

The goal is not to prove deployment readiness. The goal is to convert learned policy geometry into replay evidence: utility, regret, risk, verifier load, policy transitions, and remaining failure modes.

In [ ]:
# Notebook 71 setup
from pathlib import Path
import json, math, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Ellipse
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

SEED = 71
rng = np.random.default_rng(SEED)

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "font.size": 13,
    "axes.titlesize": 26,
    "axes.labelsize": 18,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

ACTION_ORDER = ["continue", "deepen", "verify", "stop", "fallback"]
ACTION_COLOR = {
    "continue": "#4C78A8",
    "deepen": "#F58518",
    "verify": "#B279A2",
    "stop": "#54A24B",
    "fallback": "#E45756",
}
POLICY_COLOR = {
    "threshold": "#9E9E9E",
    "learned": "#4C78A8",
    "always_continue": "#72B7B2",
    "always_verify": "#B279A2",
}

## Synthetic replay dataset

The notebook builds a deterministic synthetic replay table shaped like the output of Notebook 67. Each row is a decoding state with telemetry features, an observed action, learned policy probabilities, counterfactual rewards, and replay diagnostics.

In [ ]:
N = 4600
domains = np.array(["code", "long_context", "math", "qa", "safety"])
domain = rng.choice(domains, size=N, p=[0.22, 0.18, 0.22, 0.24, 0.14])

domain_shift = {d:i for i,d in enumerate(domains)}
is_safety = (domain == "safety").astype(float)
is_long = (domain == "long_context").astype(float)
is_code = (domain == "code").astype(float)

entropy = np.clip(rng.beta(2.8, 2.2, N)*2.15 + 0.07*rng.normal(size=N), 0.02, 2.10)
confidence = np.clip(1.03 - 0.48*entropy + 0.06*rng.normal(size=N), 0.02, 0.98)
risk_score = np.clip(0.16 + 0.50*is_safety + 0.30*(1-confidence) + 0.12*rng.normal(size=N), 0, 1)
verifier_disagreement = np.clip(0.05 + 0.55*risk_score + 0.20*rng.random(N) - 0.15*confidence, 0, 1)
latency_budget_ms = np.clip(rng.normal(650, 220, N) + 160*is_long - 120*is_code, 160, 1200)
budget_pressure = np.clip((650 - latency_budget_ms)/520 + 0.15*rng.normal(size=N), 0, 1)
confidence_margin = np.clip(np.abs(confidence - 0.5) + 0.08*rng.normal(size=N), 0, 1)
tokens_so_far = rng.integers(20, 1800, N)
step = rng.integers(1, 96, N)

# Latent best action for replay.
def best_action_row(c, e, r, d, b, dom):
    if r > 0.73 or (d > 0.68 and r > 0.35):
        return "verify"
    if b > 0.78 and c < 0.58:
        return "fallback"
    if c > 0.74 and r < 0.34 and e < 1.15:
        return "continue"
    if e > 1.05 and c < 0.62:
        return "deepen"
    if r > 0.55:
        return "verify"
    if c > 0.62:
        return "continue"
    return "deepen"

best_action = np.array([
    best_action_row(c,e,r,d,b,dom)
    for c,e,r,d,b,dom in zip(confidence, entropy, risk_score, verifier_disagreement, budget_pressure, domain)
])

# Threshold policy baseline.
threshold_action = np.where(
    (verifier_disagreement > 0.72) | (risk_score > 0.76),
    "verify",
    np.where(
        (budget_pressure > 0.78) & (confidence < 0.58),
        "fallback",
        np.where(confidence > 0.55, "continue", "deepen")
    )
)

# Learned policy approximates replay-best action but with structured errors.
learned_action = best_action.copy()
noise = rng.random(N)
learned_action[(best_action=="verify") & (noise < 0.07)] = "continue"
learned_action[(best_action=="verify") & (noise >= 0.07) & (noise < 0.12)] = "deepen"
learned_action[(best_action=="deepen") & (noise < 0.07)] = "continue"
learned_action[(best_action=="continue") & (noise < 0.04)] = "verify"
learned_action[(best_action=="stop") & (noise < 0.10)] = "verify"

observed_action = threshold_action.copy()
obs_noise = rng.random(N)
observed_action[(best_action=="verify") & (obs_noise < 0.35)] = rng.choice(["continue","deepen"], size=((best_action=="verify") & (obs_noise < 0.35)).sum())
observed_action[(best_action=="deepen") & (obs_noise < 0.18)] = "continue"
observed_action[(best_action=="fallback") & (obs_noise < 0.35)] = "verify"

# Policy probabilities: peaked around learned action with confidence-dependent softness.
probs = np.zeros((N, len(ACTION_ORDER)))
for i, a in enumerate(learned_action):
    base = np.ones(len(ACTION_ORDER)) * 0.035
    peak = 0.58 + 0.32*confidence[i] - 0.12*verifier_disagreement[i]
    peak = np.clip(peak, 0.48, 0.92)
    base[ACTION_ORDER.index(a)] = peak
    if a != "verify":
        base[ACTION_ORDER.index("verify")] += 0.10*verifier_disagreement[i] + 0.07*risk_score[i]
    if a != "deepen":
        base[ACTION_ORDER.index("deepen")] += 0.06*entropy[i]
    base = np.clip(base, 0.001, None)
    probs[i] = base/base.sum()

# Reward function.
def reward_for(action, c, e, r, d, bp, dom):
    reward = 0.55
    if action == "continue":
        reward += 0.34*c - 0.38*r - 0.12*d - 0.10*(e > 1.25)
    elif action == "deepen":
        reward += 0.18 + 0.18*e + 0.06*(c < 0.55) - 0.08*bp - 0.06*r
    elif action == "verify":
        reward += 0.20 + 0.34*r + 0.20*d - 0.16*bp
    elif action == "stop":
        reward += 0.15 + 0.20*bp - 0.18*(c > 0.60)
    elif action == "fallback":
        reward += 0.10 + 0.22*bp + 0.18*(dom == "safety") - 0.10*c
    if dom == "safety" and action == "continue" and r > 0.45:
        reward -= 0.45
    if dom == "long_context" and action == "deepen":
        reward += 0.06
    return float(np.clip(reward, 0, 1))

all_rewards = {}
for a in ACTION_ORDER:
    all_rewards[a] = np.array([
        reward_for(a, c, e, r, d, bp, dom) + 0.015*rng.normal()
        for c,e,r,d,bp,dom in zip(confidence, entropy, risk_score, verifier_disagreement, budget_pressure, domain)
    ])
    all_rewards[a] = np.clip(all_rewards[a], 0, 1)

def pick_reward(actions):
    return np.array([all_rewards[a][i] for i,a in enumerate(actions)])

reward_observed = pick_reward(observed_action)
reward_threshold = pick_reward(threshold_action)
reward_learned = pick_reward(learned_action)
reward_always_continue = all_rewards["continue"]
reward_always_verify = all_rewards["verify"]
best_reward = np.max(np.vstack([all_rewards[a] for a in ACTION_ORDER]), axis=0)

regret_threshold = best_reward - reward_threshold
regret_learned = best_reward - reward_learned
regret_always_continue = best_reward - reward_always_continue
regret_always_verify = best_reward - reward_always_verify

df = pd.DataFrame({
    "domain": domain,
    "entropy": entropy,
    "confidence": confidence,
    "risk_score": risk_score,
    "verifier_disagreement": verifier_disagreement,
    "latency_budget_ms": latency_budget_ms,
    "budget_pressure": budget_pressure,
    "confidence_margin": confidence_margin,
    "tokens_so_far": tokens_so_far,
    "step": step,
    "observed_action": observed_action,
    "best_action": best_action,
    "threshold_action": threshold_action,
    "learned_action": learned_action,
    "reward_observed": reward_observed,
    "reward_threshold": reward_threshold,
    "reward_learned": reward_learned,
    "reward_always_continue": reward_always_continue,
    "reward_always_verify": reward_always_verify,
    "regret_threshold": regret_threshold,
    "regret_learned": regret_learned,
    "regret_always_continue": regret_always_continue,
    "regret_always_verify": regret_always_verify,
})
for j,a in enumerate(ACTION_ORDER):
    df[f"p_{a}"] = probs[:,j]

df.head()

In [ ]:
def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(path)
    return path

def policy_metrics(action_col, reward_col, regret_col):
    return {
        "replay_utility": df[reward_col].mean(),
        "counterfactual_regret": df[regret_col].mean(),
        "verifier_call_rate": (df[action_col] == "verify").mean(),
        "risk_violation_rate": ((df[action_col] == "continue") & (df["risk_score"] > 0.62)).mean(),
        "fallback_rate": (df[action_col] == "fallback").mean(),
    }

metrics = {
    "observed": policy_metrics("observed_action", "reward_observed", "regret_threshold"),
    "threshold": policy_metrics("threshold_action", "reward_threshold", "regret_threshold"),
    "learned": policy_metrics("learned_action", "reward_learned", "regret_learned"),
    "always_continue": {
        "replay_utility": df["reward_always_continue"].mean(),
        "counterfactual_regret": df["regret_always_continue"].mean(),
        "verifier_call_rate": 0.0,
        "risk_violation_rate": (df["risk_score"] > 0.62).mean(),
        "fallback_rate": 0.0,
    },
    "always_verify": {
        "replay_utility": df["reward_always_verify"].mean(),
        "counterfactual_regret": df["regret_always_verify"].mean(),
        "verifier_call_rate": 1.0,
        "risk_violation_rate": 0.0,
        "fallback_rate": 0.0,
    },
}
pd.DataFrame(metrics).T

## 71.00 — Replay dataset handoff

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.axis("off")
boxes = [
    (0.08, 0.55, "Notebook 61\ntelemetry rows"),
    (0.32, 0.55, "Notebook 67\npolicy geometry"),
    (0.58, 0.55, "67 -> 71 handoff\nstate, action, reward,\npolicy probabilities"),
    (0.86, 0.55, "Notebook 71\noffline replay"),
]
for x,y,label in boxes:
    w = 0.16 if x != 0.58 else 0.22
    h = 0.28 if x != 0.58 else 0.34
    ax.add_patch(Rectangle((x-w/2, y-h/2), w, h, fill=False, lw=2))
    ax.text(x, y, label, ha="center", va="center", fontsize=16)
for (x1,_,_), (x2,_,_) in zip(boxes[:-1], boxes[1:]):
    ax.add_patch(FancyArrowPatch((x1+0.09, 0.55), (x2-0.09, 0.55), arrowstyle="->", mutation_scale=20, lw=2))
ax.text(0.5, 0.17, "Offline replay lifts learned policy geometry into deployment evidence.",
        ha="center", fontsize=18)
ax.set_title("71.00 — Replay dataset handoff", pad=20)
savefig(fig, "71_00_replay_dataset_handoff.png")
plt.show()

## 71.01 — Policy lift summary

In [ ]:
policies = ["threshold", "learned", "always_continue", "always_verify"]
labels = ["threshold", "learned", "always_continue", "always_verify"]
panels = [
    ("Replay utility", "replay_utility", "higher"),
    ("Counterfactual regret", "counterfactual_regret", "lower"),
    ("Verifier call rate", "verifier_call_rate", "bounded"),
    ("Risk violation rate", "risk_violation_rate", "bounded"),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6), sharey=False)
for ax, (title, key, direction) in zip(axes, panels):
    vals = [metrics[p][key] for p in policies]
    colors = [POLICY_COLOR[p] for p in policies]
    bars = ax.bar(range(len(policies)), vals, color=colors)
    ax.set_title(title)
    ax.set_ylim(0, max(1.0, max(vals)*1.15))
    ax.set_xticks(range(len(policies)), labels, rotation=35, ha="right")
    for b,v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v + 0.02*ax.get_ylim()[1], f"{v:.3f}", ha="center", fontsize=11)
fig.suptitle("71.01 — Policy lift summary", fontsize=28, y=1.03)
savefig(fig, "71_01_policy_lift_summary.png")
plt.show()

## 71.02 — Cost-quality frontier

In [ ]:
frontier = pd.DataFrame({
    "policy": policies,
    "verifier_call_rate": [metrics[p]["verifier_call_rate"] for p in policies],
    "replay_utility": [metrics[p]["replay_utility"] for p in policies],
})
fig, ax = plt.subplots(figsize=(9, 6.2))
for _, row in frontier.iterrows():
    p = row["policy"]
    size = 260 if p == "learned" else 150
    alpha = 1.0 if p == "learned" else 0.55
    ax.scatter(row["verifier_call_rate"], row["replay_utility"], s=size,
               color=POLICY_COLOR[p], edgecolor="black", linewidth=1.2, alpha=alpha, zorder=3)
    dx = 0.012 if p != "always_continue" else 0.006
    ax.text(row["verifier_call_rate"]+dx, row["replay_utility"]+0.006, p, fontsize=13)

tx = metrics["threshold"]["verifier_call_rate"]
ty = metrics["threshold"]["replay_utility"]
lx = metrics["learned"]["verifier_call_rate"]
ly = metrics["learned"]["replay_utility"]
ax.add_patch(FancyArrowPatch((tx, ty), (lx, ly), arrowstyle="->", mutation_scale=18, lw=2))
ax.text((tx+lx)/2 + 0.012, (ty+ly)/2 + 0.018, "policy lift", fontsize=14)

ax.set_xlabel("Verifier call rate")
ax.set_ylabel("Mean replay reward")
ax.set_xlim(-0.03, 1.08)
ax.set_ylim(0.44, 0.80)
ax.set_title("71.02 — Cost-quality frontier")
savefig(fig, "71_02_cost_quality_frontier.png")
plt.show()

## 71.03 — Replay trajectories with policy regions

In [ ]:
def region_action_grid(C, E):
    # smooth conceptual map shared with Notebook 67 design
    Z = np.full(C.shape, ACTION_ORDER.index("deepen"))
    continue_mask = E < (0.28 + 1.20*C - 0.45*C**2)
    Z[continue_mask] = ACTION_ORDER.index("continue")
    verify_mask = ((C-0.62)/0.25)**2 + ((E-1.08)/0.28)**2 < 1.0
    Z[verify_mask] = ACTION_ORDER.index("verify")
    stop_mask = ((C-0.86)/0.22)**2 + ((E-0.22)/0.23)**2 < 1.0
    Z[stop_mask] = ACTION_ORDER.index("stop")
    fallback_mask = ((C-0.05)/0.18)**2 + ((E-1.85)/0.28)**2 < 1.0
    Z[fallback_mask] = ACTION_ORDER.index("fallback")
    return Z

c_grid = np.linspace(0.02, 0.98, 160)
e_grid = np.linspace(0.03, 2.08, 160)
C, E = np.meshgrid(c_grid, e_grid)
Z = region_action_grid(C, E)
cmap = ListedColormap([ACTION_COLOR[a] for a in ACTION_ORDER])
norm = BoundaryNorm(np.arange(-0.5, len(ACTION_ORDER)+0.5), len(ACTION_ORDER))

sample_reqs = df.sample(4, random_state=9).copy()
fig, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True, sharey=True)
for ax, (_, row) in zip(axes, sample_reqs.iterrows()):
    steps = np.linspace(max(1, row.step-35), row.step+35, 9)
    c0 = np.clip(row.confidence + rng.normal(0, 0.16, len(steps)).cumsum()/8, 0.05, 0.95)
    e0 = np.clip(row.entropy + rng.normal(0, 0.25, len(steps)).cumsum()/6, 0.05, 2.05)
    r0 = np.clip(row.risk_score + rng.normal(0, 0.18, len(steps)).cumsum()/8, 0, 1)
    d0 = np.clip(row.verifier_disagreement + rng.normal(0, 0.16, len(steps)).cumsum()/8, 0, 1)
    ax.imshow(Z, origin="lower", extent=[0.02,0.98,0.03,2.08], cmap=cmap, norm=norm, alpha=0.18, aspect="auto")
    ax.plot(steps, c0, color="#4C78A8", lw=2, label="confidence")
    ax.plot(steps, r0, color="#E45756", lw=2, label="risk")
    ax.plot(steps, d0, color="#B279A2", lw=2, label="disagreement")
    for j in range(len(steps)):
        a = ACTION_ORDER[int(region_action_grid(np.array([[c0[j]]]), np.array([[e0[j]*1.2]])).item())]
        ax.scatter(steps[j], c0[j], s=55, color=ACTION_COLOR[a], edgecolor="black", linewidth=0.5, zorder=4)
    ax.set_ylabel(f"req_{int(row.name):05d}")
    ax.set_ylim(0, 1.05)
axes[-1].set_xlabel("Decode step")
axes[0].legend(loc="upper right", ncol=3)
fig.suptitle("71.03 — Replay trajectories with learned actions", fontsize=25)
savefig(fig, "71_03_replay_trajectories_with_learned_actions.png")
plt.show()

## 71.04 — Replay opportunity landscape

In [ ]:
df["learned_lift_vs_threshold"] = df["reward_learned"] - df["reward_threshold"]
sample = df.sample(1800, random_state=4)
fig, ax = plt.subplots(figsize=(10.5, 7))
sc = ax.scatter(sample["confidence"], sample["verifier_disagreement"],
                c=sample["learned_lift_vs_threshold"], cmap="coolwarm",
                s=24, alpha=0.65, edgecolor="none", vmin=-0.18, vmax=0.18)
ax.axhline(0.65, color="black", lw=1, ls="--", alpha=0.5)
ax.axvline(0.55, color="black", lw=1, ls="--", alpha=0.5)
ax.text(0.08, 0.82, "high replay\nopportunity", fontsize=16, weight="bold")
ax.text(0.70, 0.12, "routine\nexecution", fontsize=16, weight="bold")
ax.set_xlabel("Confidence")
ax.set_ylabel("Verifier disagreement")
ax.set_title("71.04 — Replay opportunity landscape")
cb = fig.colorbar(sc, ax=ax)
cb.set_label("Learned reward - threshold reward")
savefig(fig, "71_04_replay_opportunity_landscape.png")
plt.show()

## 71.05 — Safety operating slice

In [ ]:
safety = df[df["domain"] == "safety"].copy()
safe_metrics = {}
for p, action_col, reward_col in [
    ("threshold", "threshold_action", "reward_threshold"),
    ("learned", "learned_action", "reward_learned"),
    ("always_continue", None, "reward_always_continue"),
    ("always_verify", None, "reward_always_verify"),
]:
    if action_col is None:
        actions = np.array(["continue"]*len(safety)) if p == "always_continue" else np.array(["verify"]*len(safety))
    else:
        actions = safety[action_col].to_numpy()
    safe_metrics[p] = {
        "reward": safety[reward_col].mean(),
        "safe_routing_rate": 1 - ((actions == "continue") & (safety["risk_score"].to_numpy() > 0.55)).mean(),
        "verifier_rate": (actions == "verify").mean(),
    }

metrics_names = ["reward", "safe_routing_rate", "verifier_rate"]
x = np.arange(len(metrics_names))
width = 0.18
fig, ax = plt.subplots(figsize=(11.5, 6))
for i,p in enumerate(policies):
    vals = [safe_metrics[p][m] for m in metrics_names]
    ax.bar(x + (i-1.5)*width, vals, width, label=p, color=POLICY_COLOR[p])
ax.set_xticks(x, ["Replay reward", "Safe routing rate", "Verifier rate"])
ax.set_ylim(0, 1.18)
ax.legend(ncol=4, loc="upper center")
ax.set_title("71.05 — Safety operating slice")
for i,p in enumerate(policies):
    vals = [safe_metrics[p][m] for m in metrics_names]
    for j,v in enumerate(vals):
        ax.text(j + (i-1.5)*width, v+0.025, f"{v:.2f}", ha="center", fontsize=10)
savefig(fig, "71_05_safety_operating_slice.png")
plt.show()

## 71.06 — Replay lift curve

In [ ]:
fractions = np.linspace(0.05, 1.0, 20)
curves = []
# rank by opportunity; replay improves only top-k opportunity states
rank = np.argsort(-df["learned_lift_vs_threshold"].to_numpy())
base_reward = df["reward_threshold"].copy().to_numpy()
learned_reward = df["reward_learned"].to_numpy()
base_regret = df["regret_threshold"].copy().to_numpy()
learned_regret = df["regret_learned"].to_numpy()
base_risk = ((df["threshold_action"]=="continue") & (df["risk_score"] > 0.62)).astype(float).to_numpy()
learned_risk = ((df["learned_action"]=="continue") & (df["risk_score"] > 0.62)).astype(float).to_numpy()

for f in fractions:
    k = int(f*N)
    idx = rank[:k]
    reward_mix = base_reward.copy(); reward_mix[idx] = learned_reward[idx]
    regret_mix = base_regret.copy(); regret_mix[idx] = learned_regret[idx]
    risk_mix = base_risk.copy(); risk_mix[idx] = learned_risk[idx]
    curves.append({
        "replay_fraction": f,
        "utility_lift": reward_mix.mean() - base_reward.mean(),
        "regret_reduction": base_regret.mean() - regret_mix.mean(),
        "risk_change": risk_mix.mean() - base_risk.mean(),
    })
curve_df = pd.DataFrame(curves)

fig, ax = plt.subplots(figsize=(10, 6.2))
ax.plot(curve_df["replay_fraction"], curve_df["utility_lift"], marker="o", lw=2.5, label="utility lift")
ax.plot(curve_df["replay_fraction"], curve_df["regret_reduction"], marker="o", lw=2.5, label="regret reduction")
ax.plot(curve_df["replay_fraction"], curve_df["risk_change"], marker="o", lw=2.5, label="risk change")
ax.axhline(0, color="black", lw=1)
ax.set_xlabel("Replay fraction")
ax.set_ylabel("Change vs threshold policy")
ax.set_title("71.06 — Replay lift curve")
ax.legend()
ax.text(0.36, max(curve_df["utility_lift"])*0.82, "early replay captures\nmost useful corrections", fontsize=14)
savefig(fig, "71_06_replay_lift_curve.png")
plt.show()

## 71.07 — Policy transition Sankey

In [ ]:
# Lightweight Sankey-like transition diagram without extra dependencies.
trans = pd.crosstab(df["threshold_action"], df["learned_action"]).reindex(index=ACTION_ORDER, columns=ACTION_ORDER, fill_value=0)
flows = []
for i,src in enumerate(ACTION_ORDER):
    for j,dst in enumerate(ACTION_ORDER):
        count = int(trans.loc[src, dst])
        if count > 25 and src != dst:
            flows.append((src, dst, count))
flows = sorted(flows, key=lambda x: -x[2])[:10]

fig, ax = plt.subplots(figsize=(12, 7))
ax.axis("off")
left_x, right_x = 0.22, 0.78
ys = np.linspace(0.85, 0.15, len(ACTION_ORDER))
pos_left = {a:(left_x, y) for a,y in zip(ACTION_ORDER, ys)}
pos_right = {a:(right_x, y) for a,y in zip(ACTION_ORDER, ys)}
for a in ACTION_ORDER:
    ax.add_patch(Rectangle((left_x-0.12, pos_left[a][1]-0.035), 0.18, 0.07, color=ACTION_COLOR[a], alpha=0.85))
    ax.text(left_x-0.13, pos_left[a][1], a, ha="right", va="center", fontsize=14)
    ax.add_patch(Rectangle((right_x-0.06, pos_right[a][1]-0.035), 0.18, 0.07, color=ACTION_COLOR[a], alpha=0.85))
    ax.text(right_x+0.14, pos_right[a][1], a, ha="left", va="center", fontsize=14)

max_count = max(c for *_, c in flows)
for src, dst, count in flows:
    x1,y1 = pos_left[src]
    x2,y2 = pos_right[dst]
    lw = 1.0 + 9.0*count/max_count
    con = FancyArrowPatch((x1+0.07, y1), (x2-0.07, y2),
                          connectionstyle="arc3,rad=0.18", arrowstyle="-|>",
                          mutation_scale=12, lw=lw, color=ACTION_COLOR[dst], alpha=0.42)
    ax.add_patch(con)
    ax.text((x1+x2)/2, (y1+y2)/2 + 0.03, str(count), fontsize=10, ha="center", alpha=0.75)

ax.text(left_x-0.02, 0.96, "Threshold policy", fontsize=18, weight="bold", ha="center")
ax.text(right_x+0.02, 0.96, "Learned policy", fontsize=18, weight="bold", ha="center")
ax.text(0.50, 0.06, "width = transition frequency", ha="center", fontsize=13)
ax.set_title("71.07 — Policy transition map", pad=20)
savefig(fig, "71_07_policy_transition_map.png")
plt.show()

## 71.08 — Policy confidence calibration

In [ ]:
# Calibration-like diagnostic for the learned policy.
pmax = probs.max(axis=1)
agree = (learned_action == best_action).astype(float)
bins = np.linspace(0.45, 0.96, 7)
bin_ids = np.digitize(pmax, bins) - 1
xs, ys, counts = [], [], []
for b in range(len(bins)-1):
    m = bin_ids == b
    if m.sum() >= 25:
        xs.append(pmax[m].mean())
        ys.append(agree[m].mean())
        counts.append(m.sum())
fig, ax = plt.subplots(figsize=(8, 6.4))
ax.plot([0,1], [0,1], ls="--", color="#9E9E9E", lw=2, label="perfect calibration")
ax.plot(xs, ys, marker="o", color=POLICY_COLOR["learned"], lw=2.5, label="learned policy")
for x,y,c in zip(xs,ys,counts):
    ax.text(x, y+0.03, str(c), ha="center", fontsize=11)
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel("Mean learned action probability")
ax.set_ylabel("Empirical agreement with replay-best action")
ax.set_title("71.08 — Policy confidence calibration")
ax.legend(loc="upper left")
savefig(fig, "71_08_policy_confidence_calibration.png")
plt.show()

## 71.09 — Learned policy failure modes

In [ ]:
fail = df[df["learned_action"] != df["best_action"]].copy()
fail["mode"] = fail["best_action"] + " -> " + fail["learned_action"]
mode_counts = fail["mode"].value_counts().head(10).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mode_counts.index, mode_counts.values, color=POLICY_COLOR["learned"])
ax.set_xlabel("Replay states")
ax.set_ylabel("Best action -> learned action")
ax.set_title("71.09 — Learned policy failure modes")
for y,v in enumerate(mode_counts.values):
    ax.text(v+3, y, str(v), va="center", fontsize=11)
savefig(fig, "71_09_learned_policy_failure_modes.png")
plt.show()

## 71.10 — Deployment readiness gate

In [ ]:
checks = [
    ("replay utility lifts threshold", metrics["learned"]["replay_utility"] - metrics["threshold"]["replay_utility"], 0.005, "higher"),
    ("verifier rate bounded", metrics["learned"]["verifier_call_rate"], 0.35, "lower"),
    ("regret reduced", metrics["threshold"]["counterfactual_regret"] - metrics["learned"]["counterfactual_regret"], 0.005, "higher"),
    ("risk violation bounded", metrics["learned"]["risk_violation_rate"], 0.12, "lower"),
    ("fallback bounded", metrics["learned"]["fallback_rate"], 0.10, "lower"),
    ("online validation", np.nan, np.nan, "review"),
]
rows = []
for name, value, threshold, direction in checks:
    if direction == "higher":
        status = "ready" if value >= threshold else "review"
    elif direction == "lower":
        status = "ready" if value <= threshold else "review"
    else:
        status = "pending"
    rows.append((name, value, status))
readiness = sum(s=="ready" for _,_,s in rows) / len(rows)

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.axis("off")
ax.set_title("71.10 — Deployment readiness gate", pad=20)
y0 = 0.82
for i,(name,value,status) in enumerate(rows):
    y = y0 - i*0.12
    if status == "ready":
        color, mark = "#54A24B", "✓"
    elif status == "pending":
        color, mark = "#B279A2", "○"
    else:
        color, mark = "#E45756", "△"
    ax.add_patch(Rectangle((0.08, y-0.035), 0.08, 0.07, color=color, ec="black"))
    ax.text(0.12, y, mark, ha="center", va="center", color="white", fontsize=18, weight="bold")
    ax.text(0.20, y, name, va="center", fontsize=17)
    val_text = "pending" if np.isnan(value) else f"{value:.3f}"
    ax.text(0.78, y, val_text, va="center", ha="right", fontsize=16)
ax.text(0.5, 0.08, f"Engineering readiness: REVIEW  ({readiness:.0%} checks ready)",
        ha="center", va="center", fontsize=22, weight="bold",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.5))
savefig(fig, "71_10_deployment_readiness_gate.png")
plt.show()

## Export figures

In [ ]:
# Create a downloadable zip of generated figures.
zip_path = Path("notebook71_v2_figures.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIG_DIR.glob("71_*.png")):
        zf.write(p, arcname=f"figures/{p.name}")
print(f"Wrote {zip_path.resolve()}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))